Datasets:
* [openai summarize tldr](https://huggingface.co/datasets/CarperAI/openai_summarize_tldr)
* [openai summarize comparisons](https://huggingface.co/datasets/CarperAI/openai_summarize_comparisons)

Models
* [SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M)

## Imports

In [ ]:
!pip -q install torch transformers datasets rouge-score matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
from datasets import load_dataset
from functools import partial
from torch.utils.data import DataLoader
from torch.optim import AdamW
from pathlib import Path

## Starter Code

In [ ]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"  # base model for every stage
SFT_DATA = "CarperAI/openai_summarize_tldr"
PREF_DATA = "CarperAI/openai_summarize_comparisons"

MAX_PROMPT_TOKENS = 448  # post + "TL;DR:" longer than this is dropped
MAX_SUMMARY_TOKENS = 48  # summaries longer than this are dropped

SEED = 42
N_SFT_TRAIN, N_SFT_VAL = 20000, 500
N_PREF_TRAIN, N_PREF_VAL = 20000, 1000
N_EVAL_POSTS = 300

In [ ]:
def load_tokenizer(name=MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:  # SmolLM2 has no pad token
        tok.pad_token = tok.eos_token
    return tok


def load_causal_lm(name_or_path=MODEL_NAME, device=None):
    """Plain pretrained causal LM (no heads added)."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForCausalLM.from_pretrained(name_or_path)
    return model.to(device)

In [ ]:
TLDR = "TL;DR:"

def clean_post(raw_prompt):
    """Keep the post and end it with the fixed TL;DR marker."""
    post = raw_prompt.split(TLDR)[0].strip()
    return post + "\n" + TLDR


def clean_summary(raw):
    """Remove a leading marker and trailing end-of-text token."""
    text = raw.split("<|endoftext|>")[0].strip()
    if text.startswith(TLDR):
        text = text[len(TLDR):].strip()
    return " " + text


def _sample_valid(rows, n, tok, make, seed):
    order = list(range(len(rows)))
    random.Random(seed).shuffle(order)
    kept = []
    for start in range(0, len(order), 2000):
        for i in order[start:start + 2000]:
            item = make(rows[i], tok)
            if item is not None:
                kept.append(item)
                if len(kept) == n:
                    return kept
    return kept


def _make_sft(row, tok):
    prompt, summary = clean_post(row["prompt"]), clean_summary(row["label"])
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    s = tok(summary, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    if len(summary.strip()) == 0 or len(p) > MAX_PROMPT_TOKENS or len(s) > MAX_SUMMARY_TOKENS:
        return None
    return {"prompt_ids": p, "summary_ids": s, "prompt": prompt, "summary": summary}


def _make_pref(row, tok):
    prompt = clean_post(row["prompt"])
    ch, rj = clean_summary(row["chosen"]), clean_summary(row["rejected"])
    if ch == rj or not ch.strip() or not rj.strip():
        return None
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    c = tok(ch, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    r = tok(rj, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    if len(p) > MAX_PROMPT_TOKENS or max(len(c), len(r)) > MAX_SUMMARY_TOKENS:
        return None
    return {"prompt_ids": p, "chosen_ids": c, "rejected_ids": r, "prompt": prompt, "chosen": ch, "rejected": rj}


def load_sft_splits(tok):
    ds = load_dataset(SFT_DATA)
    train = _sample_valid(ds["train"], N_SFT_TRAIN, tok, _make_sft, SEED)
    val = _sample_valid(ds["valid"], N_SFT_VAL, tok, _make_sft, SEED)
    return train, val


def load_preference_splits(tok):
    ds = load_dataset(PREF_DATA)
    train = _sample_valid(ds["train"], N_PREF_TRAIN, tok, _make_pref, SEED)
    val = _sample_valid(ds["valid1"], N_PREF_VAL, tok, _make_pref, SEED)
    return train, val


def load_eval_posts(tok):
    ds = load_dataset(SFT_DATA)
    return _sample_valid(ds["test"], N_EVAL_POSTS, tok, _make_sft, SEED)

In [ ]:
def _pad(seqs, pad_id, side):
    length = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), length), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), length), dtype=torch.long)
    for i, sequence in enumerate(seqs):
        tensor = torch.tensor(sequence, dtype=torch.long)
        if side == "right":
            ids[i, :len(sequence)], mask[i, :len(sequence)] = tensor, 1
        else:
            ids[i, length - len(sequence):], mask[i, length - len(sequence):] = tensor, 1
    return ids, mask

def collate_sft(batch, pad_id):
    sequences = [item["prompt_ids"] + item["summary_ids"] for item in batch]
    ids, mask = _pad(sequences, pad_id, "right")
    prompt_lens = torch.tensor([len(item["prompt_ids"]) for item in batch])
    return {"input_ids": ids, "attention_mask": mask, "prompt_lens": prompt_lens}

def collate_pref(batch, pad_id):
    chosen, chosen_mask = _pad([item["prompt_ids"] + item["chosen_ids"] for item in batch], pad_id, "right")
    rejected, rejected_mask = _pad([item["prompt_ids"] + item["rejected_ids"] for item in batch], pad_id, "right")
    prompt_lens = torch.tensor([len(item["prompt_ids"]) for item in batch])
    return {"chosen_ids": chosen, "chosen_mask": chosen_mask, "rejected_ids": rejected, "rejected_mask": rejected_mask, "prompt_lens": prompt_lens}

def collate_prompts(batch, pad_id):
    ids, mask = _pad([item["prompt_ids"] for item in batch], pad_id, "left")
    return {"input_ids": ids, "attention_mask": mask}

In [ ]:
tok = load_tokenizer()
sft_train, sft_val = load_sft_splits(tok)
pref_train, pref_val = load_preference_splits(tok)
eval_posts = load_eval_posts(tok)

print("SFT train/val:", len(sft_train), len(sft_val))
print("Pref train/val:", len(pref_train), len(pref_val))
print("Eval posts:", len(eval_posts))

example = sft_train[0]
print("\n--- One SFT example ---")
print(example["prompt"][-300:])
print("SUMMARY:", repr(example["summary"]))

pad = tok.pad_token_id
batch = next(iter(DataLoader(sft_train, 8, shuffle=True, collate_fn=partial(collate_sft, pad_id=pad))))
print("\nSFT batch:", tuple(batch["input_ids"].shape), "prompt_lens", batch["prompt_lens"].tolist())

batch = next(iter(DataLoader(pref_train, 8, shuffle=True, collate_fn=partial(collate_pref, pad_id=pad))))
print("Pref batch:", tuple(batch["chosen_ids"].shape), tuple(batch["rejected_ids"].shape))

batch = next(iter(DataLoader(eval_posts, 8, collate_fn=partial(collate_prompts, pad_id=pad))))
print("Prompt batch:", tuple(batch["input_ids"].shape), "(left-padded)")

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

... (20 outputs left out)

Generating test split:   0%|          | 0/83629 [00:00<?, ? examples/s]

Generating valid1 split:   0%|          | 0/33082 [00:00<?, ? examples/s]

Generating valid2 split:   0%|          | 0/50715 [00:00<?, ? examples/s]

SFT train/val: 20000 500
Pref train/val: 20000 1000
Eval posts: 300

--- One SFT example ---
n thirty minutes. My little sister was in a black car with no A/C in mid-70 degree weather for thirty minutes. She's was still alive, thankfully, but redfaced and crying.

"Crud," my innocent mind thought.

Suffice to say that I didn't leave the house for a while, let alone to go to the park.
TL;DR:
SUMMARY: ' Me and my sister were going to the park, put my sister in her carseat, left her there for thirty minutes, mom saved her from my ignorance, got grounded for a month.'

SFT batch: (8, 464) prompt_lens [420, 279, 350, 324, 254, 318, 375, 428]
Pref batch: (8, 455) (8, 434)
Prompt batch: (8, 378) (left-padded)


### Task 1: Run the starter code
* 1.1 Set up the environment and run check starter.py (Appendix A.6).
* 1.2 Report the size of every subset, and the mean and maximum prompt and summary length
in tokens for the SFT train subset.
* 1.3 Write one helper you will use in every stage, token logprobs(model, input ids,
attention mask), returning
```
log π(xt | x<t) for every position t ≥ 1, shape (B,L −1),
```
and a summary mask that is 1 exactly on the summary tokens (not prompt, not padding).
Check it on one batch by printing the decoded tokens where the mask is 1.

In [ ]:
mean_prompt_length = sum(len(example['prompt']) for example in sft_train) / len(sft_train)
print("Mean prompt length in SFT train:", mean_prompt_length)

max_prompt_length = max(len(example['prompt']) for example in sft_train)
print("Maximum prompt length in SFT train:", max_prompt_length)

mean_summary_length = sum(len(example['summary']) for example in sft_train) / len(sft_train)
print("Mean summary length in SFT train:", mean_summary_length)

max_summary_length = max(len(example['summary']) for example in sft_train)
print("Maximum summary length in SFT train:", max_summary_length)

Mean prompt length in SFT train: 1275.68515
Maximum prompt length in SFT train: 2137
Mean summary length in SFT train: 139.8802
Maximum summary length in SFT train: 271


In [ ]:
def token_logprobs(model, input_ids, attention_mask, prompt_lens):

    logits = model(input_ids=input_ids,attention_mask=attention_mask,).logits
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    shift_attention = attention_mask[:, 1:].bool()

    log_probs = torch.log_softmax(shift_logits, dim=-1) # torch.Size([B, L - 1])
    token_log_probs = torch.gather(log_probs,dim=-1,index=shift_labels.unsqueeze(-1),).squeeze(-1)
    summary_mask = torch.zeros_like(shift_labels, dtype=torch.bool)

    for i, prompt_len in enumerate(prompt_lens.tolist()):
        start = prompt_len - 1
        summary_mask[i, start:] = shift_attention[i, start:]

    return token_log_probs, summary_mask

In [ ]:
sft_batch = next(iter(DataLoader(sft_train,batch_size=8,shuffle=True,collate_fn=partial(collate_sft,pad_id=tok.pad_token_id,),)))

model = load_causal_lm()
model.eval()
device = next(model.parameters()).device

with torch.no_grad():
    log_probs, summary_mask = token_logprobs(
        model,
        sft_batch["input_ids"].to(device),
        sft_batch["attention_mask"].to(device),
        sft_batch["prompt_lens"],
    )

print("Log probabilities shape:", log_probs.shape)
print("Summary mask shape:", summary_mask.shape)

for i in range(sft_batch["input_ids"].size(0)):
    summary_token_ids = sft_batch["input_ids"][i, 1:][summary_mask[i].cpu()]
    print(f"Example {i} summary tokens:")
    print(tok.decode(summary_token_ids.tolist()))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Log probabilities shape: torch.Size([8, 435])
Summary mask shape: torch.Size([8, 435])
Example 0 summary tokens:
 GF hates being touched and im not sure how to make her more comfortable or if it will ever get better. Any advice (other than telling me to leave her) is welcome<|endoftext|>
Example 1 summary tokens:
 I am a jerk who unknowingly crossed an emotional boundary and  cheated on husband (which is reprehensible, however, we are now okay) and possibly caused damage to someone else's relationship.<|endoftext|>
Example 2 summary tokens:
 girl has boyfriend but wants my dick. I'm desperate so i want to give her the dick. My morals are not to give her the dick but im desperate at the moment so...<|endoftext|>
Example 3 summary tokens:
 I don't want to move in with my GF but she thinks its a good idea and I'm worried that my hesitance is just an unreasonable fear of commitment.<|endoftext|>
Example 4 summary tokens:
 Fucked up my relationship, trying to mend it through kind gestures a

### Task 2: SFT

In [ ]:
def sft_loss(model, batch):
    device = next(model.parameters()).device

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    prompt_lens = batch["prompt_lens"].to(device)

    token_log_probs, summary_mask = token_logprobs(
        model,
        input_ids,
        attention_mask,
        prompt_lens,
    )

    summary_mask = summary_mask.to(token_log_probs.device)

    return -token_log_probs[summary_mask].mean()

In [ ]:
def evaluate_sft(model, dataloader):
    model.eval()
    total_loss = 0.0
    batches = 0

    with torch.no_grad():
        for batch in dataloader:
            total_loss += sft_loss(model, batch).item()
            batches += 1

    return total_loss / batches

In [ ]:
BATCH_SIZE = 4
LEARNING_RATE = 5e-5
GRAD_CLIP_NORM = 1.0
LOG_EVERY = 250

sft_train_loader = DataLoader(
    sft_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=partial(collate_sft, pad_id=tok.pad_token_id),
)

sft_val_loader = DataLoader(
    sft_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=partial(collate_sft, pad_id=tok.pad_token_id),
)

model = load_causal_lm()
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
def generate_summaries(model, examples, tokenizer, n=5):
    model.eval()
    device = next(model.parameters()).device
    summaries = []

    with torch.no_grad():
        for example in examples[:n]:
            input_ids = torch.tensor(
                [example["prompt_ids"]],
                dtype=torch.long,
                device=device,
            )
            attention_mask = torch.ones_like(input_ids)

            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=48,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            new_tokens = generated[0, input_ids.shape[1]:]
            summaries.append(tokenizer.decode(new_tokens, skip_special_tokens=True))

    return summaries


before_sft = generate_summaries(model, eval_posts, tok, n=5)

print("Summaries before SFT:")
for i, summary in enumerate(before_sft, start=1):
    print(f"{i}. {summary}")

Summaries before SFT:
1.  I'm not going to tell him I'm sorry, I'm not going to tell him I'm sorry, I'm not going to tell him I'm sorry, I'm not going to tell him I'm sorry, I'm not going
2.  I'm not sure if I should ask her to be my girlfriend or not.

I'm not sure if I should ask her to be my girlfriend.

I'm not sure if I should ask her to be my girlfriend.
3.  I am not able to love her anymore.

I am not able to love her anymore.

I am not able to love her anymore.

I am not able to love her anymore.

I am not able
4.  I'm not sure what to do. I'm not sure if I should keep my savings account or not. I'm not sure if I should keep my credit card or not. I'm not sure if I should keep my checking account or
5.  I am not sure if I should hold out or not. I am not sure if I should be able to talk to her again.

I am not sure if I should be able to talk to her again. I am not sure


In [ ]:
from tqdm.notebook import tqdm

model.train()
global_step = 0

pbar = tqdm(sft_train_loader, desc="Training SFT")
for batch in pbar:
    optimizer.zero_grad(set_to_none=True)

    loss = sft_loss(model, batch)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=GRAD_CLIP_NORM,
    )

    optimizer.step()
    global_step += 1

    # Update progress bar description with current training loss
    pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    if global_step % LOG_EVERY == 0:
        validation_loss = evaluate_sft(model, sft_val_loader)

        print(
            f"Step {global_step}/{len(sft_train_loader)} | "
            f"train loss: {loss.item():.4f} | "
            f"validation loss: {validation_loss:.4f}"
        )

        model.train()

Training SFT:   0%|          | 0/5000 [00:00<?, ?it/s]

Step 250/5000 | train loss: 3.0938 | validation loss: 2.5190
Step 500/5000 | train loss: 2.4375 | validation loss: 2.4982
Step 750/5000 | train loss: 1.9766 | validation loss: 2.4863
Step 1000/5000 | train loss: 2.5000 | validation loss: 2.4787
Step 1250/5000 | train loss: 2.6719 | validation loss: 2.4731
Step 1500/5000 | train loss: 2.5000 | validation loss: 2.4671
Step 1750/5000 | train loss: 2.3594 | validation loss: 2.4637
Step 2000/5000 | train loss: 2.6406 | validation loss: 2.4601
Step 2250/5000 | train loss: 2.5938 | validation loss: 2.4583
Step 2500/5000 | train loss: 2.3125 | validation loss: 2.4540
Step 2750/5000 | train loss: 2.1250 | validation loss: 2.4527
Step 3000/5000 | train loss: 3.1406 | validation loss: 2.4498
Step 3250/5000 | train loss: 2.7812 | validation loss: 2.4483
Step 3500/5000 | train loss: 2.0938 | validation loss: 2.4467
Step 3750/5000 | train loss: 1.9297 | validation loss: 2.4459
Step 4000/5000 | train loss: 2.2500 | validation loss: 2.4439
Step 4250/5

In [ ]:
checkpoint_dir = Path("checkpoints/sft")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(checkpoint_dir)
tok.save_pretrained(checkpoint_dir)

print(f"SFT model saved to {checkpoint_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SFT model saved to checkpoints/sft


In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Define the destination directory in your Google Drive
drive_dest_dir = Path('/content/drive/MyDrive/checkpoints/sft')
drive_dest_dir.mkdir(parents=True, exist_ok=True)

# Source checkpoint directory
source_dir = Path('checkpoints/sft')

# Copy each file in the checkpoint directory to Google Drive
for file_path in source_dir.glob('*'):
    if file_path.is_file():
        shutil.copy2(file_path, drive_dest_dir / file_path.name)

print(f"Checkpoint files successfully uploaded to Google Drive at: {drive_dest_dir}")

Mounted at /content/drive
Checkpoint files successfully uploaded to Google Drive at: /content/drive/MyDrive/checkpoints/sft


### DPO

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from google.colab import drive

drive.mount('/content/drive')
# local path: "checkpoints/sft"
# Drive path: "/content/drive/MyDrive/checkpoints/sft"
checkpoint_path = "/content/drive/MyDrive/checkpoints/sft"


tok = AutoTokenizer.from_pretrained(checkpoint_path)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
policy = AutoModelForCausalLM.from_pretrained(checkpoint_path).to(device)
refrence = AutoModelForCausalLM.from_pretrained(checkpoint_path).to(device)

print(f"Successfully reloaded SFT models and tokenizer from: {checkpoint_path} to {device}")

Mounted at /content/drive


Loading weights:   0%|          | 0/272 [00:01<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Successfully reloaded SFT models and tokenizer from: /content/drive/MyDrive/checkpoints/sft to cuda


### DPO Helper Functions & Loss
We will define the token-level log probabilities extractor and the DPO loss function.

In [ ]:
import torch.nn.functional as F

# Redefine calculate_DPO_loss to use F.logsigmoid instead of torch.logsigmoid
def calculate_DPO_loss(pref_log_prob, dispref_log_prob, pref_log_prob_ref, dispref_log_prob_ref, beta=0.1):
    # Calculate relative log probabilities
    pref_relative_logprob = pref_log_prob - pref_log_prob_ref
    dispref_relative_logprob = dispref_log_prob - dispref_log_prob_ref

    # DPO formulation using F.logsigmoid
    logits = beta * (pref_relative_logprob - dispref_relative_logprob)
    loss = -F.logsigmoid(logits).mean()

    # Compute implicit rewards and metrics for logging
    reward_margins = beta * (pref_relative_logprob - dispref_relative_logprob)
    reward_accuracies = (reward_margins > 0).float().mean()

    return loss, pref_relative_logprob, dispref_relative_logprob, reward_accuracies, reward_margins

### Custom DPO Collate Function
To match your training loop's expectation of keys like `chosen_pairs_ids`, `rejected_pairs_ids`, `chosen_loss_mask`, and `rejected_loss_masks`.

In [ ]:
from tqdm.notebook import tqdm

def train(model, ref_model, optimizer, train_data, val_data, epochs=1):
  model.train()
  ref_model.eval()

  for epoch in range(epochs):
    pbar = tqdm(train_data, desc=f"Epoch {epoch+1}/{epochs}")
    for batch in pbar:
      # Use the keys generated by collate_pref
      chosen_ids = batch['chosen_ids'].to(model.device)
      chosen_mask = batch['chosen_mask'].to(model.device)
      rejected_ids = batch['rejected_ids'].to(model.device)
      rejected_mask = batch['rejected_mask'].to(model.device)
      prompt_lens = batch['prompt_lens'].to(model.device)

      # Forward pass for policy model
      pref_log_prob, chosen_loss_mask = token_logprobs(model, chosen_ids, chosen_mask, prompt_lens)
      dispref_log_prob, rejected_loss_mask = token_logprobs(model, rejected_ids, rejected_mask, prompt_lens)

      # Forward pass for reference model
      with torch.no_grad():
        pref_log_prob_ref, _ = token_logprobs(ref_model, chosen_ids, chosen_mask, prompt_lens)
        dispref_log_prob_ref, _ = token_logprobs(ref_model, rejected_ids, rejected_mask, prompt_lens)

      # We sum the log probabilities of the selected summary tokens
      pref_log_prob_sum = (pref_log_prob * chosen_loss_mask).sum(dim=-1)
      dispref_log_prob_sum = (dispref_log_prob * rejected_loss_mask).sum(dim=-1)
      pref_log_prob_ref_sum = (pref_log_prob_ref * chosen_loss_mask).sum(dim=-1)
      dispref_log_prob_ref_sum = (dispref_log_prob_ref * rejected_loss_mask).sum(dim=-1)

      # Calculate DPO Loss using your calculate_DPO_loss function
      loss, pref_relative_logprob, dispref_relative_logprob, reward_accuracies, reward_margins = calculate_DPO_loss(
          pref_log_prob_sum, dispref_log_prob_sum, pref_log_prob_ref_sum, dispref_log_prob_ref_sum, beta=0.5
      )

      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

      pbar.set_postfix({"loss": f"{loss.item():.4f}", "accuracy": f"{reward_accuracies.item():.2f}"})

### Start DPO Training
Create dataloaders and initialize the optimizer to start training.

In [ ]:
from torch.optim import AdamW

DPO_BATCH_SIZE = 4
LEARNING_RATE_DPO = 5e-6

# Utilizing your predefined collate_pref
train_loader = DataLoader(
    pref_train,
    batch_size=DPO_BATCH_SIZE,
    shuffle=True,
    collate_fn=partial(collate_pref, pad_id=tok.pad_token_id)
)

# Reference model does not need gradients
for param in refrence.parameters():
    param.requires_grad = False

opt = AdamW(policy.parameters(), lr=LEARNING_RATE_DPO)

In [ ]:
# Start DPO Training using the updated loss function
train(policy, refrence, opt, train_loader, val_data=None, epochs=1)

Epoch 1/1:   0%|          | 0/5000 [00:00<?, ?it/s]